# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook provides a step-by-step template for loading and exploring the [FAIR² Mental Health Survey Dataset from Kilifi County, Kenya](https://sen.science/doi/10.71728/senscience.vcs2-05nj) using the `mlcroissant` Python library. The dataset is published with full Croissant AI-Ready data standards and is accessible via a Croissant schema URL.

### Dataset Source
The dataset source is provided via a Croissant schema URL ([https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json](https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json)).


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant pandas matplotlib

## 1. Data Loading
Load the dataset schema metadata and records from the dataset using `mlcroissant`.


In [ ]:
import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset name: {getattr(metadata, 'name', '<No name>')}")
print(f"Description: {getattr(metadata, 'description', '<No description>')}")

## 2. Data Overview
Review available record sets, fields, and their IDs. All entities and references use their `@id` for clarity and reproducibility.


In [ ]:
# List all record sets, their @id, and fields
record_sets = metadata.record_sets()
print(f"Number of record sets: {len(record_sets)}\n")
for i, record_set in enumerate(record_sets):
    print(f"Record set {i+1}: @id = {record_set.id}")
    print(f"  Name: {getattr(record_set, 'name', '<No name>')}")
    fields = getattr(record_set, 'fields', lambda: [])()
    print(f"  Number of fields: {len(fields)}")
    for field in fields:
        print(f"    - Field: @id = {field.id}, Name: {getattr(field, 'name', '<No name>')}, Data type: {getattr(field, 'data_type', '<No data_type>')}")
    print()

## 3. Data Extraction
Extract all records for each record set. Data will be loaded into Pandas DataFrames for quick inspection. Replace or reference entities by their `@id` at all steps. 


In [ ]:
dataframes = {}
record_set_ids = [rs.id for rs in metadata.record_sets()]
print(f"Loading records for record sets: {record_set_ids}\n")

# Load all record sets into separate dataframes
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Record set: {record_set_id}, Rows: {df.shape[0]}, Columns: {df.columns.tolist()}")
    print(df.head(2))


## 4. Exploratory Data Analysis (EDA)
Apply standard processing to the main survey data record set. The main record set typically contains a field for mental health assessment scores, such as GAD-7 total score. We'll:
- Filter records with valid scores
- Normalize a numeric field (e.g., `gad7_total_score`)
- Group data by a demographic (e.g., `gender`)

Replace `<record_set_id>`, `<numeric_field_id>`, and `<group_field>` with actual `@id` values as found in the overview above. For demonstration, we will choose the record set and field IDs programmatically for the dataset as published.


In [ ]:
# Select the main survey record set (should be the first or named for the survey)
if len(record_set_ids) == 0:
    raise ValueError('No record sets found in the dataset.')

# For this dataset, we will use the first record set as the main data
main_record_set_id = record_set_ids[0]
df = dataframes[main_record_set_id]

# Attempt to find numeric and group fields based on common field names
possible_numeric_fields = [col for col in df.columns if (('total' in col.lower() and 'score' in col.lower()) or ('score' in col.lower())) and df[col].dtype in [int,float,'int64','float64']]
if len(possible_numeric_fields) == 0:
    # Fall back to first float/int column
    possible_numeric_fields = [col for col in df.columns if df[col].dtype in [int,float,'int64','float64']]
if len(possible_numeric_fields) == 0:
    raise ValueError('No numeric field found in main record set.')
numeric_field = possible_numeric_fields[0]

# For grouping, look for gender or similar
possible_group_fields = [col for col in df.columns if 'gender' in col.lower() or 'sex' in col.lower() or 'group' in col.lower()]
group_field = possible_group_fields[0] if possible_group_fields else df.columns[0]

print(f"Using main_record_set_id: {main_record_set_id}")
print(f"Numeric field selected: {numeric_field}")
print(f"Group field selected: {group_field}")

# Apply filter (threshold for normalized GAD-7/PHQ-9 scores, e.g., >7)
threshold = 7
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold}: {filtered_df.shape[0]}")
print(filtered_df[[numeric_field, group_field]].head())

# Normalize the numeric field
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"\nFirst 5 normalized {numeric_field} values:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Group and aggregate
if group_field in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field)[numeric_field, f"{numeric_field}_normalized"].mean()
    print(f"\nGrouped data by {group_field} (mean values):\n", grouped_df.head())

## 5. Visualization
Visualize the distribution of the main numeric field (e.g., GAD-7/PHQ-9/PSQ score) and group means by demographic (e.g., gender).


In [ ]:
# Histogram of total scores
plt.figure(figsize=(7,4))
df[numeric_field].hist(bins=15, color='skyblue', edgecolor='black')
plt.title(f'Distribution of {numeric_field}')
plt.xlabel(numeric_field)
plt.ylabel('Count')
plt.show()

# Boxplot of total scores by group
if group_field in df.columns:
    plt.figure(figsize=(6,4))
    df.boxplot(column=numeric_field, by=group_field, grid=False)
    plt.title(f'{numeric_field} by {group_field}')
    plt.suptitle('')
    plt.xlabel(group_field)
    plt.ylabel(numeric_field)
    plt.show()


## 6. Conclusion
This notebook demonstrated how to load, inspect, and process a Croissant-AI-ready survey dataset using `mlcroissant`. We overviewed its structure, extracted records, performed EDA including filtering, normalization, grouping, and visualization. The approach can be generalized to other Croissant-compliant datasets by referencing entities and fields using their `@id`s, as illustrated above.
